# LOGISTIC REGRESSION GENE-SET BURDEN SCORES 

In [ ]:
# Load packages
import os
import re
import gc
import glob
import pandas as pd 
from biomart import BiomartServer
from functools import reduce
import subprocess
import numpy as np
import statsmodels.api as sm

# Set workspace environment variables
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-9f879fc4"
os.environ["ARTIFACT_REGISTRY_DOCKER_REPO"] = (
    "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)

# Load the remaining workspace environment settings
%run Setting_Env_Variables.ipynb

In [ ]:
# =============================================================================================================
# Set up directories
# =============================================================================================================

OUTPUT_DIR = "burden_polygenic_score"
SUMMARY_DIR = "logistic_summary"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SUMMARY_DIR, exist_ok=True)

WORKSPACE_BUCKET = os.environ["WORKSPACE_BUCKET"]

GOOGLE_CLOUD_PROJECT = os.environ["GOOGLE_CLOUD_PROJECT"]

# =============================================================================================================
# Copy files from Google Cloud Storage
# =============================================================================================================

# Copy gene-set burden score files
subprocess.run(
    f"gsutil cp {WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_gene_sets/somatic_mutations/*remove*PRS.tsv {OUTPUT_DIR}/",
    shell=True,
    check=True
)

# Copy phenotype file
subprocess.run(
    f"gsutil cp {WORKSPACE_BUCKET}/data/saige_phenotype_files/aml_phenotype_files/EUR_phenotype_file.txt {OUTPUT_DIR}/",
    shell=True,
    check=True
)

# Copy exome burden file
subprocess.run(
    f"gsutil cp {WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_gene_sets/exome/* {OUTPUT_DIR}/",
    shell=True,
    check=True
)

# Copy genomic metrics file
subprocess.run(
    f"cp /home/dataproc/workspace/vwb-aou-datasets-controlled-v8r9/v8/wgs/short_read/snpindel/aux/qc/genomics_metrics_Dec142023_1859_02_tz0000.tsv {OUTPUT_DIR}/",
    shell=True,
    check=True
)

# Copy pca file
subprocess.run(
    f"gsutil cp {WORKSPACE_BUCKET}/data/pcs/EUR_flashpca.pcs {OUTPUT_DIR}/",
    shell=True,
    check=True
)

# =============================================================================================================
# List gene-set burden score files
# =============================================================================================================

files = sorted(
    glob.glob(os.path.join(OUTPUT_DIR, "*remove*PRS.tsv"))
)

print(f"Found {len(files)} gene-set burden score files.")

# =============================================================================================================
# Load files necessary for logistic regression
# =============================================================================================================

# Load phenotype file
pheno = pd.read_csv(
    os.path.join(OUTPUT_DIR, "EUR_phenotype_file.txt"),
    sep="\t"
)

# Load exome burden file
exome = pd.read_csv(
    os.path.join(OUTPUT_DIR, "exome-burden-PRS.tsv"),
    sep="\t"
)

# Load genomic metric file
metrics = pd.read_csv(
    os.path.join(OUTPUT_DIR, "genomics_metrics_Dec142023_1859_02_tz0000.tsv"),
    sep="\t"
)

# Format exome burden file
exome = (
    exome.rename(columns={
        "IID": "person_id",
        "EXOME_SCORE1_SUM": "exome_burden_prs"
    })
    [["person_id", "exome_burden_prs"]]
)

# Format genomic metric file
metrics = (
    metrics.rename(columns={
        "research_id": "person_id"
    })
    [["person_id", "sample_source"]]
)

# Fitler for unrelated subset 
eur_flashpca = pd.read_csv(
    "burden_polygenic_score/EUR_flashpca.pcs",
    sep="\t"
)[["IID", "group"]].rename(
    columns={"IID": "person_id"}
)

eur_flashpca = eur_flashpca[
    eur_flashpca["group"] == "Unrelated_Inliers"
][['person_id']].copy()

# =============================================================================================================
# Logistic regression of gene sets for exome burden polygenic risk score
# =============================================================================================================

# Define the covariates 
covariates = [
    "sex", "age",
    "PC1", "PC2", "PC3", "PC4", "PC5",
    "PC6", "PC7", "PC8", "PC9", "PC10"
]

for file in files:

    gene_set = os.path.basename(file).replace("-PRS.tsv", "")

    print(f"\nProcessing file: {gene_set} ...")

    # Load gene-set burden score
    score = pd.read_csv(file, sep="\t")

    # Format gene-set burden score
    score = (
        score.rename(columns={
            "IID": "person_id",
            "TOTAL_SCORE1_SUM": "burden_prs"
        })
        [["person_id", "burden_prs"]]
    )

    # Merge all data frames
    merged_df = (
        pheno
        .merge(score, on="person_id", how="inner")
        .merge(exome, on="person_id", how="inner")
        .merge(metrics, on="person_id", how="inner")
        .merge(eur_flashpca, on="person_id", how="inner")
    )

    print(f"Number of individuals: {len(merged_df)}")

    # Skip gene sets with no rare variants
    if merged_df["burden_prs"].max() == 0:

        print(f"Skipping file: {gene_set} (no rare variants)")

        continue

    # Standardize burden scores
    merged_df["burden_prs"] = (
        merged_df["burden_prs"] - merged_df["burden_prs"].mean()
    ) / merged_df["burden_prs"].std()

    merged_df["exome_burden_prs"] = (
        merged_df["exome_burden_prs"] - merged_df["exome_burden_prs"].mean()
    ) / merged_df["exome_burden_prs"].std()

    # Define outcome and predictors
    X = merged_df[
        ["burden_prs", "exome_burden_prs"] + covariates
    ]

    y = merged_df["aml_phenotype"]

    # Add intercept
    X = sm.add_constant(X)

    # Run logistic regression
    logistic = sm.Logit(y, X).fit(disp=False)

    # Extract regression coefficients and statistics
    logistic_summary = pd.DataFrame({
        "Coefficient": logistic.params,
        "Std_Error": logistic.bse,
        "z_value": logistic.tvalues,
        "p_value": logistic.pvalues,
        "OR": np.exp(logistic.params),
        "CI_lower": np.exp(logistic.conf_int()[0]),
        "CI_upper": np.exp(logistic.conf_int()[1])
    })
    
    print(logistic_summary)

    # Save summary
    output_file = os.path.join(
        SUMMARY_DIR,
        f"{gene_set}-unrelated-logistic-summary.tsv"
    )

    logistic_summary.to_csv(
        output_file,
        sep="\t",
        index=True,
        index_label="Variable"
    )

    print(f"Saved results: {output_file}")

# Copy exome burden file
subprocess.run(
    f"gsutil cp {OUTPUT_DIR}/* {WORKSPACE_BUCKET}/data/gene_burden_analysis/results/",
    shell=True,
    check=True
)

print("\nFinished processing all gene sets.")